# Taller 4 - Experimentación con MLflow (Palmer Penguins)

Flujo del notebook:

1. Leer los datos **crudos** desde `postgres-data` (`raw.penguins`, cargada por `postgres-init/01-raw.sql`).
2. Preprocesar y guardar los datos **procesados** en `postgres-data` (`processed.penguins`).
3. Leer los procesados **desde la base de datos** y separar train/test.
4. **(TODO)** Configurar MLflow y correr **≥ 20 ejecuciones** variando hiperparámetros.
5. **(TODO)** Registrar el mejor modelo en el Model Registry y asignarle el alias que usa la API.

Las secciones 1-3 reutilizan lo hecho en taller002 (features de pingüinos) y taller003 (raw → clean en BD).

## 0. Configuración

In [1]:
import os

import mlflow
import pandas as pd
from sqlalchemy import create_engine, text
from sklearn.model_selection import train_test_split

# Inyectadas por docker-compose.yaml (servicio jupyter).
DATA_DB_URI = os.environ["DATA_DB_URI"]
MLFLOW_TRACKING_URI = os.environ["MLFLOW_TRACKING_URI"]
MODEL_NAME = os.environ.get("MODEL_NAME", "penguins-classifier")
MODEL_ALIAS = os.environ.get("MODEL_ALIAS", "champion")
EXPERIMENT_NAME = "penguinsTaller004"

engine = create_engine(DATA_DB_URI)

print(f"MLflow:  {MLFLOW_TRACKING_URI}")
print(f"Modelo a registrar: {MODEL_NAME}@{MODEL_ALIAS}")

MLflow:  http://mlflow:5000
Modelo a registrar: penguins-classifier@champion


## 1. Datos crudos (desde la base de datos)

`raw.penguins` se llena una sola vez al crear el volumen de `postgres-data`, con el CSV de `data/penguins.csv`
**sin limpiar** (los `NA` quedan como `NULL`).

In [2]:
df_raw = pd.read_sql("SELECT * FROM raw.penguins ORDER BY rowid", con=engine)
print(df_raw.shape)
print(df_raw.isna().sum())
df_raw.head()

(344, 9)
rowid                 0
species               0
island                0
bill_length_mm        2
bill_depth_mm         2
flipper_length_mm     2
body_mass_g           2
sex                  11
year                  0
dtype: int64


,rowid,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,year
0,1,Adelie,Torgersen,39.1,18.7,181.0,3750.0,male,2007
1,2,Adelie,Torgersen,39.5,17.4,186.0,3800.0,female,2007
2,3,Adelie,Torgersen,40.3,18.0,195.0,3250.0,female,2007
3,4,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN,2007
4,5,Adelie,Torgersen,36.7,19.3,193.0,3450.0,female,2007


## 2. Preprocesamiento → `processed.penguins`

Misma limpieza que taller003 / taller002:

- Se quedan las columnas que usa el modelo (`FEATURE_COLUMNS`) + el target (`species`).
- Se eliminan filas con faltantes en esas columnas.
- `rowid` se conserva para poder rastrear cada fila procesada hasta su fila cruda.

In [3]:
FEATURE_COLUMNS = ["island", "bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g", "sex"]
NUMERIC_COLUMNS = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
CATEGORICAL_COLUMNS = ["island", "sex"]
TARGET = "species"

df_processed = (
    df_raw[["rowid", *FEATURE_COLUMNS, TARGET]]
    .dropna(subset=[*FEATURE_COLUMNS, TARGET])
    .reset_index(drop=True)
)
df_processed[NUMERIC_COLUMNS] = df_processed[NUMERIC_COLUMNS].astype(float)

# replace: re-ejecutar el notebook deja la tabla consistente con la limpieza actual.
df_processed.to_sql("penguins", con=engine, schema="processed", if_exists="replace", index=False)

print(f"Crudos: {len(df_raw)} filas -> procesados: {len(df_processed)} filas")
df_processed.head()

Crudos: 344 filas -> procesados: 333 filas


,rowid,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,species
0,1,Torgersen,39.1,18.7,181.0,3750.0,male,Adelie
1,2,Torgersen,39.5,17.4,186.0,3800.0,female,Adelie
2,3,Torgersen,40.3,18.0,195.0,3250.0,female,Adelie
3,5,Torgersen,36.7,19.3,193.0,3450.0,female,Adelie
4,6,Torgersen,39.3,20.6,190.0,3650.0,male,Adelie


## 3. Leer los procesados desde la base de datos y separar train/test

In [4]:
df = pd.read_sql("SELECT * FROM processed.penguins ORDER BY rowid", con=engine)

X = df[FEATURE_COLUMNS]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(y.value_counts())

Train: (266, 6), Test: (67, 6)
species
Adelie       146
Gentoo       119
Chinstrap     68
Name: count, dtype: int64


## 4. Configurar MLflow

- Apuntar el cliente al servidor (`MLFLOW_TRACKING_URI`).
- Crear/seleccionar el experimento.

In [6]:
mlflow.set_tracking_uri

<function mlflow.tracking._tracking_service.utils.set_tracking_uri(uri: str | pathlib.Path) -> None>

In [7]:
mlflow.set_experiment(EXPERIMENT_NAME)

<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1791051439227, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1791051439227, lifecycle_stage='active', name='penguinsTaller004', tags={}, trace_location=None, workspace='default'>

In [8]:
MLFLOW_TRACKING_URI,

('http://mlflow:5000',)

## 5. Función de entrenamiento + registro de un run

In [10]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

# 1. Qué hacer con cada grupo de columnas.
#    Cada entrada de la lista es: (nombre_que_tu_eliges, transformador, columnas)
preprocess = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_COLUMNS),   # ¿qué lista de columnas?
    ("num", "passthrough", NUMERIC_COLUMNS),                            # ¿y aquí?
])

# 2. Encadenar preprocesamiento + modelo.
#    Cada paso es: (nombre, objeto)
pipeline = Pipeline([
    ("preprocess", preprocess),
    ("model", RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)),
])

# 3. Entrenar: UN solo fit para todo el pipeline
pipeline.fit(X_train, y_train)

# 4. Predecir sobre los datos que el modelo NO vio
y_pred = pipeline.predict(X_test)

# 5. Métricas: (valor_real, valor_predicho)
acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average="macro")
print(f"accuracy={acc:.4f}  f1_macro={f1:.4f}")

accuracy=1.0000  f1_macro=1.0000


In [11]:
import mlflow
from sklearn.model_selection import StratifiedKFold, cross_val_score
from mlflow.models import infer_signature

def train_and_log(params, run_name):
    with mlflow.start_run(run_name=run_name) as run:          # opcional: un nombre legible, ej. "rf-prueba"
        pipeline = Pipeline([
            ("preprocess", preprocess),
            ("model", RandomForestClassifier(**params, random_state=42, n_jobs=-1)),
        ])
        pipeline.fit(X_train, y_train)
        y_pred = pipeline.predict(X_test)
    
        metrics = {
            "accuracy": accuracy_score(y_test, y_pred),
            "f1_macro": f1_score(y_test, y_pred, average="macro"),
        }    
    
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        
        cv_scores = cross_val_score(
            pipeline,                     # ¿qué se valida? (el objeto completo, sin entrenar o entrenado, da igual: lo clona)
            X_train, y_train,                # ¿con qué datos? (pista: NO los de test)
            cv=cv,
            scoring="f1_macro",
            n_jobs=-1
        )
        
        metrics["cv_f1_macro_mean"] = cv_scores.mean()
        metrics["cv_f1_macro_std"] = cv_scores.std()
    
    
        mlflow.log_params(params)     # registrar parámetros
        mlflow.log_metrics(metrics)    # registrar métricas
    
        signature = infer_signature(X_train, pipeline.predict(X_train))   # (ejemplo de entrada, ejemplo de salida)
    
        mlflow.sklearn.log_model(
            sk_model=pipeline,
            name="model",
            signature=signature,
            input_example=X_train.head(3),
            skops_trusted_types=["sklearn.tree._tree.Tree"],   # ← ¿qué tipo exacto? (está en el error)
        )
        return run.info.run_id, metrics

run_id, metrics = train_and_log(params={"n_estimators": 50, "max_depth": 3}, run_name="rf-prueba2")
print(run_id, metrics)

2026/10/04 14:59:12 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


🏃 View run rf-prueba2 at: http://mlflow:5000/#/experiments/1/runs/ac603e133145441cab8fc4980df77653
🧪 View experiment at: http://mlflow:5000/#/experiments/1
ac603e133145441cab8fc4980df77653 {'accuracy': 1.0, 'f1_macro': 1.0, 'cv_f1_macro_mean': np.float64(0.9716708183833443), 'cv_f1_macro_std': np.float64(0.024046132677754767)}


## 6. Experimentación: al menos 20 ejecuciones

In [13]:
from itertools import product

N_ESTIMATORS = [1, 5, 20, 100, 300]     # de "casi nada" a "mucho", con saltos crecientes
MAX_DEPTH = [1, 2, 4, 8, None]          # None = sin límite de profundidad

grid = [
    {"n_estimators": n, "max_depth": d}   # ← n y d: los valores de ESTA vuelta
    for n, d in product(N_ESTIMATORS, MAX_DEPTH)
]
print(len(grid))   # 25
grid[:3]

25


[{'n_estimators': 1, 'max_depth': 1},
 {'n_estimators': 1, 'max_depth': 2},
 {'n_estimators': 1, 'max_depth': 4}]

In [18]:
results = []
for params in grid:
    run_name = f"rf-n{params['n_estimators']}-d{params['max_depth']}"   # ← único por run
    run_id, metrics = train_and_log(params=params, run_name=run_name)
    results.append({"run_id": run_id, "run_name": run_name, **params, **metrics})
    print(f"{run_name:<14} cv_f1={metrics['cv_f1_macro_mean']:.4f}  test_f1={metrics['f1_macro']:.4f}")
    
df_results = pd.DataFrame(results).sort_values(
    ["cv_f1_macro_mean", "cv_f1_macro_std"],   # primero mejor media; si empatan, menor desviación
    ascending=[False, True],
)
df_results

2026/10/04 15:05:46 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


🏃 View run rf-n1-d1 at: http://mlflow:5000/#/experiments/1/runs/25ae4926d1684a08b5150f0332ecfd03
🧪 View experiment at: http://mlflow:5000/#/experiments/1
rf-n1-d1       cv_f1=0.5038  test_f1=0.4679


2026/10/04 15:05:49 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


🏃 View run rf-n1-d2 at: http://mlflow:5000/#/experiments/1/runs/7423ff47b37340ee95fc26343ebdb4e8
🧪 View experiment at: http://mlflow:5000/#/experiments/1
rf-n1-d2       cv_f1=0.9152  test_f1=0.9271


2026/10/04 15:05:52 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


...

2026/10/04 15:06:55 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


🏃 View run rf-n300-d8 at: http://mlflow:5000/#/experiments/1/runs/56077afa8cf740bfa260c9d6bfc208bd
🧪 View experiment at: http://mlflow:5000/#/experiments/1
rf-n300-d8     cv_f1=0.9909  test_f1=1.0000


2026/10/04 15:06:59 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


🏃 View run rf-n300-dNone at: http://mlflow:5000/#/experiments/1/runs/7bc4fe2944bf46af9db0bf7980bae651
🧪 View experiment at: http://mlflow:5000/#/experiments/1
rf-n300-dNone  cv_f1=0.9909  test_f1=1.0000


,run_id,run_name,n_estimators,max_depth,accuracy,f1_macro,cv_f1_macro_mean,cv_f1_macro_std
18,43f00c1805fa48c8b69b42294c5dc2a1,rf-n100-d8,100,8.0,1.000000,1.000000,0.990872,0.011180
19,04fa5fd81a034c40b98a337416e666f7,rf-n100-dNone,100,NaN,1.000000,1.000000,0.990872,0.011180
23,56077afa8cf740bfa260c9d6bfc208bd,rf-n300-d8,300,8.0,1.000000,1.000000,0.990872,0.011180
24,7bc4fe2944bf46af9db0bf7980bae651,rf-n300-dNone,300,NaN,1.000000,1.000000,0.990872,0.011180
8,15430c54fadd4e4a96f580975a85dd6e,rf-n5-d8,5,8.0,0.910448,0.908126,0.986705,0.010859
9,dceca8d2ad73457087bbf4b96c56dffa,rf-n5-dNone,5,NaN,0.910448,0.908126,0.983301,0.008734
13,539ba8f5c0004219b34eb4a409ddf37e,rf-n20-d8,20,8.0,0.985075,0.987258,0.983301,0.008734
14,950003f0e68a418a8ff282af1f785e13,rf-n20-dNone,20,NaN,0.985075,0.987258,0.983301,0.008734
12,b9fc8f1d371f4cabb4beb2b7a5da8a42,rf-n20-d4,20,4.0,0.985075,0.987258,0.978722,0.014415
7,24631bd4ee844914849be4e61cf0ec53,rf-n5-d4,5,4.0,0.970149,0.974396,0.973313,0.008130


## 7. Elegir el mejor run y registrarlo en el Model Registry

In [24]:
ranking = mlflow.search_runs(
    experiment_names=[EXPERIMENT_NAME],
    order_by=["metrics.cv_f1_macro_mean DESC", "metrics.cv_f1_macro_std ASC"],   # ← metrics., no params.
    max_results=5,                                                                # ← 5 para ver el empate
)
ranking[["run_id", "tags.mlflow.runName", "params.n_estimators", "params.max_depth",
         "metrics.cv_f1_macro_mean", "metrics.cv_f1_macro_std"]]

,run_id,tags.mlflow.runName,params.n_estimators,params.max_depth,metrics.cv_f1_macro_mean,metrics.cv_f1_macro_std
0,7bc4fe2944bf46af9db0bf7980bae651,rf-n300-dNone,300,None,0.990872,0.01118
1,56077afa8cf740bfa260c9d6bfc208bd,rf-n300-d8,300,8,0.990872,0.01118
2,04fa5fd81a034c40b98a337416e666f7,rf-n100-dNone,100,None,0.990872,0.01118
3,43f00c1805fa48c8b69b42294c5dc2a1,rf-n100-d8,100,8,0.990872,0.01118
4,a44da5d8c75d4a56905604ad752fea33,rf-n300-dNone,300,None,0.990872,0.01118


In [30]:
# Desempate: entre los empatados, el más simple (menos árboles, profundidad acotada).
chosen = mlflow.search_runs(
    experiment_names=[EXPERIMENT_NAME],
    filter_string="params.n_estimators = '100' and params.max_depth = '8'",
    order_by=["attributes.start_time DESC"],   # ← el más reciente primero
    max_results=1,
)
best_run_id = chosen.iloc[0]["run_id"]
best_run_name = chosen.iloc[0]["tags.mlflow.runName"]

In [29]:
chosen

,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.cv_f1_macro_mean,metrics.accuracy,metrics.cv_f1_macro_std,metrics.f1_macro,params.n_estimators,params.max_depth,tags.mlflow.user,tags.mlflow.source.type,tags.mlflow.runName,tags.mlflow.source.name
0,43f00c1805fa48c8b69b42294c5dc2a1,1,FINISHED,mlflow-artifacts:/1/43f00c1805fa48c8b69b42294c...,2026-10-04 15:06:34.074000+00:00,2026-10-04 15:06:37.103000+00:00,0.990872,1.0,0.01118,1.0,100,8,root,NOTEBOOK,rf-n100-d8,experimentos_mlflow.ipynb


In [32]:
model_version = mlflow.register_model(
    model_uri=f"runs:/{best_run_id}/model",   # el ID que acabas de obtener + el `name` que usaste en log_model
    name=MODEL_NAME,
)
print(model_version.name, model_version.version)

Registered model 'penguins-classifier' already exists. Creating a new version of this model...
2026/10/04 15:23:24 WARNING mlflow.tracking._model_registry.fluent: Run with id 43f00c1805fa48c8b69b42294c5dc2a1 has no artifacts at artifact path 'model', registering model based on models:/m-1202186e752d4fa184e72a13d598c784 instead
2026/10/04 15:23:24 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: penguins-classifier, version 1


penguins-classifier 1


Created version '1' of model 'penguins-classifier'.


In [34]:
from mlflow import MlflowClient

client = MlflowClient()
client.set_registered_model_alias(
    name=MODEL_NAME,
    alias=MODEL_ALIAS,                       # variable definida en la sección 0
    version=model_version.version,       # mira lo que imprimiste: "penguins-classifier 1"
)

## 8. Verificar: cargar el modelo desde MLflow igual que lo hará la API

In [35]:
loaded_model = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME}@{MODEL_ALIAS}")

predicciones = loaded_model.predict(X_test.head())
predicciones

array(['Gentoo', 'Chinstrap', 'Adelie', 'Gentoo', 'Gentoo'], dtype=object)